In [ ]:
# ---- SETTINGS: this is the only cell you edit ----

# 1. The connection string of the ORDERS Eventstream (the same one as in part 1).
CONNECTION_STRING = "PASTE-YOUR-CONNECTION-STRING-HERE"

# 2. The connection string of the WEATHER Eventstream (the new one from this part).
#    Where to find it: open the Eventstream, click the source box (the custom endpoint),
#    then in the pane below: Event Hub > SAS Key Authentication > "Connection string-primary key".
WEATHER_CONNECTION_STRING = "PASTE-THE-WEATHER-CONNECTION-STRING-HERE"

ORDERS_PER_MINUTE = 20     # about 20 orders a minute, before the weather adds its own
MUTE_REGION = None         # to stage an outage, write a region: "North America", "Europe", "Asia Pacific" or "Latin America"
RUN_MINUTES = 120          # the store and the weather feed stop by themselves after this many minutes
SEND_WEATHER_HISTORY = True    # the first run sends 28 days of weather history; set this to False after that run

In [ ]:
# ---- HELPERS + (on the first run only) 28 days of REAL hourly weather for the 17 cities ----
# Weather data by Open-Meteo.com (CC BY 4.0, free for non-commercial use, no key).
import base64, hashlib, hmac, json, time, urllib.parse, urllib.request
from datetime import datetime, timezone

CITIES = {  # city: (latitude, longitude, region)
    "Vancouver": (49.28, -123.12, "North America"), "Seattle": (47.61, -122.33, "North America"),
    "Denver": (39.74, -104.99, "North America"), "Toronto": (43.65, -79.38, "North America"),
    "New York": (40.71, -74.01, "North America"), "London": (51.51, -0.13, "Europe"),
    "Berlin": (52.52, 13.41, "Europe"), "Oslo": (59.91, 10.75, "Europe"), "Madrid": (40.42, -3.70, "Europe"),
    "Zurich": (47.38, 8.54, "Europe"), "Tokyo": (35.68, 139.69, "Asia Pacific"),
    "Sydney": (-33.87, 151.21, "Asia Pacific"), "Auckland": (-36.85, 174.76, "Asia Pacific"),
    "Singapore": (1.35, 103.82, "Asia Pacific"), "Santiago": (-33.45, -70.67, "Latin America"),
    "Bogota": (4.71, -74.07, "Latin America"), "Sao Paulo": (-23.55, -46.63, "Latin America")}
CONDITIONS = {0: "Clear", 1: "Mostly clear", 2: "Partly cloudy", 3: "Overcast", 45: "Fog", 48: "Fog", 51: "Light drizzle",
              53: "Drizzle", 55: "Heavy drizzle", 61: "Light rain", 63: "Rain", 65: "Heavy rain", 71: "Light snow",
              73: "Snow", 75: "Heavy snow", 80: "Light showers", 81: "Showers", 82: "Heavy showers", 95: "Thunderstorm"}


def endpoint(connection_string, name):
    conn = dict(part.split("=", 1) for part in connection_string.strip().split(";") if "=" in part)
    if "Endpoint" not in conn or "EntityPath" not in conn:
        raise ValueError(f"Paste the connection string into {name} (the first cell), then run the cells again.")
    conn["url"] = "https://" + conn["Endpoint"].replace("sb://", "").strip("/") + "/" + conn["EntityPath"]
    return conn


def send(conn, events):
    uri = urllib.parse.quote_plus(conn["url"])
    expiry = str(int(time.time()) + 3600)
    signature = base64.b64encode(hmac.new(conn["SharedAccessKey"].encode(), f"{uri}\n{expiry}".encode(), hashlib.sha256).digest()).decode()
    token = f"SharedAccessSignature sr={uri}&sig={urllib.parse.quote_plus(signature)}&se={expiry}&skn={conn['SharedAccessKeyName']}"
    body = json.dumps([{"Body": json.dumps(e)} for e in events]).encode("utf-8")
    request = urllib.request.Request(conn["url"] + "/messages?timeout=60&api-version=2014-01", data=body, method="POST")
    request.add_header("Authorization", token)
    request.add_header("Content-Type", "application/vnd.microsoft.servicebus.json")
    with urllib.request.urlopen(request, timeout=30) as response:
        return response.status


def weather_service(parameters):
    latitudes = ",".join(str(c[0]) for c in CITIES.values())
    longitudes = ",".join(str(c[1]) for c in CITIES.values())
    url = f"https://api.open-meteo.com/v1/forecast?latitude={latitudes}&longitude={longitudes}&timezone=UTC&{parameters}"
    with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "fabric-demo"}), timeout=60) as response:
        return json.loads(response.read())


def reading(city, time_text, temp, precipitation, snow, wind, code, minutes):
    latitude, longitude, region = CITIES[city]
    precipitation = precipitation or 0.0
    return {"ReadingTime": time_text + ":00Z", "City": city, "Region": region, "Latitude": latitude, "Longitude": longitude,
            "TempC": temp, "PrecipMm": precipitation, "RainRate": round(precipitation * 60 / minutes, 2),   # millimetres per hour
            "SnowCm": snow or 0.0, "WindKmh": wind, "WeatherCode": code, "Condition": CONDITIONS.get(code, str(code)),
            "IntervalMin": minutes, "Source": "open-meteo.com"}


weather = endpoint(WEATHER_CONNECTION_STRING, "WEATHER_CONNECTION_STRING")
if SEND_WEATHER_HISTORY:
    data = weather_service("hourly=temperature_2m,precipitation,snowfall,weather_code,wind_speed_10m&past_days=28&forecast_days=1")
    now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:00")
    history = []
    for city, place in zip(CITIES, data):
        h = place["hourly"]
        for i, t in enumerate(h["time"]):
            if t <= now and h["temperature_2m"][i] is not None:
                history.append(reading(city, t, h["temperature_2m"][i], h["precipitation"][i], h["snowfall"][i],
                                       h["wind_speed_10m"][i], h["weather_code"][i], 60))
    for start in range(0, len(history), 300):
        send(weather, history[start:start + 300])
    rainy = sum(1 for r in history if r["RainRate"] >= 0.4)
    print(f"Sent {len(history)} hourly readings. {rainy} of them are rainy hours.")
    print("Now set SEND_WEATHER_HISTORY = False in the first cell, so the next run does not send the history again.")
else:
    print("Weather history: skipped (SEND_WEATHER_HISTORY is False).")

In [ ]:
# ---- THE STORE + THE WEATHER FEED: run this cell. It runs until RUN_MINUTES pass, or until you press Stop. ----
# (Run the cell above it first: it defines the cities and the helper functions.)
import math, random

PRODUCTS = [  # id, name, category, price, how popular
    (1, "Summit 2P Tent", "Camping", 331, 94), (2, "Basecamp 6P Tent", "Camping", 667, 50),
    (3, "Arctic Sleeping Bag", "Camping", 217, 70), (4, "Trailhead 45L Pack", "Hiking", 179, 120),
    (5, "Ridge Trekking Poles", "Hiking", 85, 85), (6, "Alpine Hiking Boots", "Hiking", 208, 104),
    (7, "Storm Shell Jacket", "Apparel", 247, 101), (8, "Thermal Base Layer", "Apparel", 75, 133),
    (9, "Summit Fleece", "Apparel", 122, 114), (10, "Glacier Water Bottle", "Accessories", 33, 154),
    (11, "Headlamp Pro", "Accessories", 56, 120), (12, "Trail First-Aid Kit", "Accessories", 43, 92)]
REGIONS = {"North America": (1, 39), "Europe": (2, 27), "Asia Pacific": (3, 16), "Latin America": (4, 9)}   # id, share of orders
PAYMENTS = [("Card", 62), ("PayPal", 20), ("Apple Pay", 12), ("Bank transfer", 6)]
QUANTITIES = [(1, 70), (2, 22), (3, 8)]
CHANNELS = [("Web", 64), ("App", 36)]
CURVE = [0.62, 0.55, 0.50, 0.48, 0.50, 0.56, 0.66, 0.78, 0.90, 1.00, 1.08, 1.14,
         1.20, 1.26, 1.32, 1.38, 1.42, 1.45, 1.42, 1.34, 1.20, 1.02, 0.84, 0.70]
# What the SIMULATED shoppers do with the REAL weather in their city:
RAIN, HEAVY_RAIN, COLD, HOT = 0.4, 2.0, 5.0, 26.0             # mm per hour, mm per hour, degrees C, degrees C
EFFECTS = {"rain": (["Storm Shell Jacket"], 3.0), "heavy rain": (["Storm Shell Jacket"], 4.0),
           "cold": (["Thermal Base Layer", "Summit Fleece"], 1.8), "hot": (["Glacier Water Bottle"], 2.2)}
CITY_NAMES = {region: [c for c, v in CITIES.items() if v[2] == region] for region in REGIONS}
TOTAL_POPULAR = sum(p[4] for p in PRODUCTS)
TOTAL_SHARE = sum(r[1] for r in REGIONS.values())


def pick(pairs):
    return random.choices([p[0] for p in pairs], weights=[p[1] for p in pairs])[0]


def order(product=None, city=None):
    pid, name, category, price, _ = product or random.choices(PRODUCTS, weights=[p[4] for p in PRODUCTS])[0]
    if city is None:
        region = random.choices(list(REGIONS), weights=[r[1] for r in REGIONS.values()])[0]
        city = random.choice(CITY_NAMES[region])
    region = CITIES[city][2]
    quantity = pick(QUANTITIES)
    now = datetime.now(timezone.utc)
    return {"OrderId": f"ORD-{int(now.timestamp() * 1000)}-{random.randint(100, 999)}",
            "OrderTime": now.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z",
            "ProductId": pid, "Product": name, "Category": category, "RegionId": REGIONS[region][0], "Region": region,
            "City": city, "Quantity": quantity, "UnitPrice": price, "SalesAmount": quantity * price,
            "PaymentMethod": pick(PAYMENTS), "Channel": pick(CHANNELS)}


def weather_now():
    data = weather_service("current=temperature_2m,precipitation,snowfall,weather_code,wind_speed_10m")
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M")
    readings = []
    for city, place in zip(CITIES, data):
        c = place["current"]
        readings.append(reading(city, stamp, c["temperature_2m"], c["precipitation"], c["snowfall"], c["wind_speed_10m"],
                                c["weather_code"], int(c["interval"] // 60)))
    return readings


def what_weather_does(r):
    tags = ["heavy rain"] if r["RainRate"] >= HEAVY_RAIN else ["rain"] if r["RainRate"] >= RAIN else []
    if r["TempC"] is not None and r["TempC"] < COLD:
        tags.append("cold")
    if r["TempC"] is not None and r["TempC"] > HOT:
        tags.append("hot")
    return tags


def how_many(expected):                     # a random whole number around "expected"
    limit, count, p = math.exp(-expected), 0, 1.0
    while expected > 0:
        p *= random.random()
        if p <= limit:
            break
        count += 1
    return count


orders = endpoint(CONNECTION_STRING, "CONNECTION_STRING")
print("The store is open, and the weather feed is on." + (f" Muted region: {MUTE_REGION}." if MUTE_REGION else ""))
sent = extra = 0
latest, last_sent, last_fetch, started = {}, {}, 0.0, time.time()
while time.time() - started < RUN_MINUTES * 60:
    tick = time.time()
    if tick - last_fetch >= 300:                                    # every 5 minutes: ask the weather service
        try:
            readings = weather_now()
            changed = [r for r in readings
                       if last_sent.get(r["City"], (None, 0))[0] != (r["TempC"], r["PrecipMm"], r["WeatherCode"], r["WindKmh"])
                       or tick - last_sent[r["City"]][1] > 870]     # send when the values changed, or every 15 minutes
            if changed:
                send(weather, changed)
                for r in changed:
                    last_sent[r["City"]] = ((r["TempC"], r["PrecipMm"], r["WeatherCode"], r["WindKmh"]), tick)
            latest = {r["City"]: r for r in readings}
            raining = [c for c, r in latest.items() if r["RainRate"] >= RAIN]
            print(f"{datetime.now(timezone.utc):%H:%M:%S} UTC  orders: {sent} + {extra} because of the weather | "
                  f"weather readings sent: {len(changed)} | raining now: {', '.join(raining) or 'nowhere'}")
        except Exception as problem:                                # a failed call must not stop the store
            print("weather call failed, will try again:", str(problem)[:120])
        last_fetch = tick
    now = datetime.now(timezone.utc)
    busy = CURVE[now.hour] * (1.12 if now.weekday() >= 5 else 1.0)
    per_batch = ORDERS_PER_MINUTE * busy / 6                         # one batch every 10 seconds
    batch = [order() for _ in range(max(1, round(per_batch * random.uniform(0.6, 1.4))))]
    weather_orders = []
    for city, r in latest.items():                                  # the weather-driven extra orders
        city_share = REGIONS[CITIES[city][2]][1] / TOTAL_SHARE / len(CITY_NAMES[CITIES[city][2]])
        for tag in what_weather_does(r):
            names, times = EFFECTS[tag]
            for product in (p for p in PRODUCTS if p[1] in names):
                expected = per_batch * city_share * product[4] / TOTAL_POPULAR * (times - 1)
                weather_orders += [order(product, city) for _ in range(how_many(expected))]
    batch = [o for o in batch if o["Region"] != MUTE_REGION]
    weather_orders = [o for o in weather_orders if o["Region"] != MUTE_REGION]
    if batch or weather_orders:
        send(orders, batch + weather_orders)
        sent += len(batch)
        extra += len(weather_orders)
    time.sleep(max(0.0, 10 - (time.time() - tick)))
print("The store is closed. Orders sent:", sent, "+", extra, "because of the weather")